In [0]:
import re
import pandas as pd
from pathlib import Path

# Caminho para o arquivo de dados
CAMINHO = str(Path('../data/DataCoSupplyChainDataset.csv').resolve())

colunas = [
    'Shipping Mode',
    'Delivery Status',
    'Late_delivery_risk',
    'Days for shipping (real)',
    'Days for shipment (scheduled)',
    'Customer City',
    'Order Region'
]
tabela = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "ISO-8859-1")
    .csv(CAMINHO)
)

# Delta não aceita espaços, parênteses e vírgulas nos nomes de coluna
tabela = tabela.toDF(*[re.sub(r'[^0-9a-zA-Z_]+', '_', c).strip('_') for c in tabela.columns])

tabela.write.format("Delta").mode("overwrite").saveAsTable("tabela")

display(tabela)



In [0]:
%sql
SELECT 
    Shipping_Mode,
    SUM(CASE WHEN Delivery_Status = 'Late delivery' THEN 1 ELSE 0 END) AS Atrasado,
    COUNT(Shipping_Mode) AS Total,
    Round((SUM(CASE WHEN Delivery_Status = 'Late delivery' THEN 1 ELSE 0 END) * 100.0 / COUNT(Shipping_Mode)),2) AS `% Atraso`
FROM 
    tabela
WHERE 
    Delivery_Status <> 'Shipping canceled'
GROUP BY 
    Shipping_Mode

ORDER BY `% Atraso` DESC


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW enviados AS
SELECT *,
       CASE WHEN Delivery_Status = 'Late delivery' THEN 1 ELSE 0 END AS atrasado,
       Days_for_shipping_real - Days_for_shipment_scheduled          AS diferenca,
       MONTH(TO_TIMESTAMP(order_date_DateOrders, 'M/d/yyyy H:mm'))        AS mes,
       WEEKDAY(TO_TIMESTAMP(order_date_DateOrders, 'M/d/yyyy H:mm')) + 1  AS dia_semana
FROM   tabela
WHERE  Delivery_Status <> 'Shipping canceled';

In [0]:
%sql
SELECT Order_Status, Delivery_Status, COUNT(*) AS pedidos
FROM   tabela
WHERE  Delivery_Status = 'Shipping canceled'
GROUP BY Order_Status, Delivery_Status
ORDER BY pedidos DESC

In [0]:
%sql
SELECT COUNT(*) AS total_pedidos,
       SUM(atrasado) AS atrasados,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados

In [0]:
%sql
SELECT Shipping_Mode,
       COUNT(*) AS total_pedidos,
       SUM(atrasado) AS atrasados,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados
GROUP BY Shipping_Mode
ORDER BY taxa_atraso_pct DESC

In [0]:
%sql
SELECT Shipping_Mode,
       ROUND(AVG(Days_for_shipment_scheduled), 2) AS prazo_prog_medio,
       ROUND(AVG(Days_for_shipping_real), 2)      AS prazo_real_medio,
       ROUND(AVG(diferenca), 2)                   AS diferenca_media,
       MIN(diferenca)                             AS menor_diferenca,
       MAX(diferenca)                             AS maior_diferenca
FROM   enviados
GROUP BY Shipping_Mode
ORDER BY diferenca_media DESC

In [0]:
%sql
SELECT Shipping_Mode,
       Days_for_shipment_scheduled AS dias_prog,
       Days_for_shipping_real      AS dias_real,
       COUNT(*) AS pedidos
FROM   enviados
GROUP BY Shipping_Mode, Days_for_shipment_scheduled, Days_for_shipping_real
ORDER BY Shipping_Mode, dias_real

In [0]:
%sql
SELECT Days_for_shipping_real AS dias_real,
       COUNT(*) AS pedidos,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_dos_pedidos,
       MAX(Days_for_shipment_scheduled) AS prazo_prometido
FROM   enviados
WHERE  Shipping_Mode = 'Standard Class'
GROUP BY Days_for_shipping_real
ORDER BY dias_real

In [0]:
%sql
SELECT Order_Region, COUNT(*) AS total_pedidos,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados
GROUP BY Order_Region
ORDER BY taxa_atraso_pct DESC

In [0]:
%sql
SELECT Customer_State, COUNT(*) AS total_pedidos,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados
GROUP BY Customer_State
HAVING COUNT(*) >= 300
ORDER BY taxa_atraso_pct DESC
LIMIT 10

In [0]:
%sql
SELECT mes, COUNT(*) AS total_pedidos,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados
GROUP BY mes
ORDER BY mes

In [0]:
%sql
SELECT dia_semana, COUNT(*) AS total_pedidos,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados
GROUP BY dia_semana
ORDER BY dia_semana

In [0]:
%sql
SELECT dia_semana, COUNT(*) AS total_pedidos,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados
GROUP BY dia_semana
ORDER BY dia_semana

In [0]:
%sql
SELECT Order_Status, COUNT(*) AS total_pedidos,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados
GROUP BY Order_Status
ORDER BY taxa_atraso_pct DESC

In [0]:
%sql
SELECT Customer_Segment, COUNT(*) AS total_pedidos,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados
GROUP BY Customer_Segment
ORDER BY taxa_atraso_pct DESC

In [0]:
%sql
SELECT Market, COUNT(*) AS total_pedidos,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados
GROUP BY Market
ORDER BY taxa_atraso_pct DESC

In [0]:
%sql
SELECT Category_Name, COUNT(*) AS total_pedidos, SUM(atrasado) AS atrasados,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados
GROUP BY Category_Name
HAVING COUNT(*) >= 100
ORDER BY taxa_atraso_pct DESC
LIMIT 10

In [0]:
%sql
SELECT Order_Item_Quantity, COUNT(*) AS total_pedidos,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados
GROUP BY Order_Item_Quantity
ORDER BY Order_Item_Quantity

In [0]:
%sql
SELECT quartil,
       MIN(Sales) AS vendas_min,
       MAX(Sales) AS vendas_max,
       COUNT(*)   AS total_pedidos,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM (
    SELECT atrasado, Sales, NTILE(4) OVER (ORDER BY Sales) AS quartil
    FROM   enviados
) t
GROUP BY quartil
ORDER BY quartil

In [0]:
%sql
SELECT Market, Customer_Segment, COUNT(*) AS total_pedidos,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM   enviados
WHERE  Shipping_Mode = 'Standard Class'
GROUP BY Market, Customer_Segment
ORDER BY Market, Customer_Segment